![Banner](../Image/04_DeepCuaslaML.png)

# 4.6 Counterfactual / Potential Outcomes Models

This notebook introduces the **Potential Outcomes** framework and implements three practical deep learning models for counterfactual time-series prediction:

1. **DeepSynth (Neural Synthetic Control)**
2. **CRN (Counterfactual Recurrent Network)**
3. **G-Net (Deep G-Computation)**

The workflow includes:
- theory and estimands (ATE, ATT, CATE, time-varying effects),
- data construction from sector ETF returns,
- model training in PyTorch,
- counterfactual inference and causal effect estimation.

## Part 1 - Theoretical Foundation

### The Potential Outcomes Framework

For binary treatment $T \in \{0,1\}$, define potential outcomes for unit $i$:

$$Y_i(1) = 	ext{outcome unit } i 	ext{ would have under treatment}$$
$$Y_i(0) = 	ext{outcome unit } i 	ext{ would have under control}$$

The **Individual Treatment Effect (ITE)** is:

$$	au_i = Y_i(1) - Y_i(0)$$

The fundamental problem of causal inference: for each unit/time, we only observe one potential outcome (the factual one). The other is counterfactual and must be estimated.

In time series with treatment history $\bar{T}_t = (T_1,\ldots,T_t)$:
 
$$Y_t(\bar{a}) = \text{outcome at time } t \text{ under treatment history } \bar{a}$$

### Key Estimands

| Estimand | Formula | Question |
|---|---|---|
| ATE | $\mathbb{E}[Y(1)-Y(0)]$ | Average causal effect in population |
| ATT | $\mathbb{E}[Y(1)-Y(0)\mid T=1]$ | Effect on treated units |
| CATE | $\mathbb{E}[Y(1)-Y(0)\mid X=x]$ | Effect conditional on covariates |
| Time-varying ATE | $\mathbb{E}[Y_t(\bar{a}) - Y_t(\bar{a}')]$ | Effect of treatment strategy over time |

### The Confounding Problem in Time Series

Time-varying confounders create feedback loops:

$$X_t 
ightarrow T_t 
ightarrow Y_{t+1} 
ightarrow X_{t+1} 
ightarrow T_{t+1} 
ightarrow \cdots$$

Here $X_t$ can be both:
- a confounder (affects treatment and outcome), and
- an intermediate variable (affected by prior treatment).

This breaks naive regression assumptions and motivates causal sequence models.

## Part 2 - Model Intuition

### 1) DeepSynth / Neural Synthetic Control

Classical synthetic control estimates a treated unit's counterfactual using weighted controls:

$$\hat{Y}_t^{(0)} = \sum_{j \in 	ext{controls}} w_j Y_{t,j}, \quad w_j \ge 0,\; \sum_j w_j = 1$$

DeepSynth replaces fixed linear weights with neural representation learning + attention.

### 2) CRN - Counterfactual Recurrent Network

CRN uses recurrent representations and adversarial balancing to reduce treatment-assignment bias:

$$\min_{	ext{enc, pred}} \max_D \; \mathcal{L}_{	ext{pred}} - \lambda\mathcal{L}_{	ext{disc}}$$

### 3) G-Net - Deep G-Computation

G-Net learns sequential generative models for covariates and outcomes and rolls forward under interventions:

$$\mathbb{E}[Y_t(\bar{a})] = \int \prod_{k=1}^{t} p(X_k\mid\bar{X}_{k-1},\bar{A}_{k-1}=\bar{a}_{k-1})\,p(Y_t\mid\bar{X}_t,\bar{A}_t=\bar{a}_t)\,d\bar{X}$$

### 1) Install Dependencies (Optional)


In [ ]:
import importlib
import subprocess
import sys

PACKAGES = [
    "numpy", "pandas", "scipy", "torch", "scikit-learn",
    "matplotlib", "seaborn", "networkx", "yfinance",
]

for pkg in PACKAGES:
    mod = "sklearn" if pkg == "scikit-learn" else pkg
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import pydeepcausalml  # noqa: F401
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/zia207/PyDeepCausalML.git"]
    )

import pydeepcausalml
print("pydeepcausalml", pydeepcausalml.__version__, "ready.")


### 2) Import Libraries and Set Device


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from pydeepcausalml import set_seed

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

set_seed(42)
run_fast = True

import yfinance as yf
from sklearn.preprocessing import StandardScaler
from pydeepcausalml import DeepSynth, CRN, GNet, counterfactual_model


### 3) Download Market Data and Define Treatment/Outcome


In [ ]:
TICKERS = {
    "XLF": "Financials",
    "XLE": "Energy",
    "XLK": "Technology",
    "XLV": "Healthcare",
    "XLI": "Industrials",
    "XLY": "ConsumerDisc",
    "XLP": "ConsumerStap",
    "XLU": "Utilities",
}

import yfinance as yf

raw = yf.download(
    list(TICKERS.keys()),
    start="2018-01-01",
    end="2024-01-01",
    auto_adjust=True,
    progress=False,
)["Close"]

returns = np.log(raw / raw.shift(1)).dropna()
returns.columns = [TICKERS[t] for t in returns.columns]

if returns.shape[0] < 50:
    print("yfinance unavailable; using synthetic demo data.")
    rng = np.random.default_rng(42)
    T, d = 1500, len(TICKERS)
    VAR_NAMES = list(TICKERS.values())
    market = rng.normal(0.0, 0.8, size=(T, 1)).astype(np.float64)
    idio = rng.normal(0.0, 0.6, size=(T, d)).astype(np.float64)
    loading = np.linspace(0.5, 1.1, d, dtype=np.float64)[None, :]
    data_np = (market @ loading + idio).astype(np.float64)
else:
    data_np = returns.values.astype(np.float64)
    T, d = data_np.shape
    VAR_NAMES = returns.columns.tolist()

print(f"Shape: {data_np.shape}")
print(f"Variables ({d}): {VAR_NAMES}")
print(f"Time steps: {T}")

# Treatment: Technology shock (top quartile daily return)
tech_idx = VAR_NAMES.index("Technology")
fin_idx = VAR_NAMES.index("Financials")
tech_ret = data_np[:, tech_idx]
threshold = np.quantile(tech_ret, 0.75)
treatment = (tech_ret >= threshold).astype(np.float64)
outcome = data_np[:, fin_idx]

print(f"Treatment rate (Technology shock): {treatment.mean():.3f}")


### 4) Build Counterfactual Time-Series Dataset


In [ ]:
LAG = 20
EPOCHS = 40 if run_fast else 80
DEVICE = "cpu"
BATCH_SIZE = 64
LR = 3e-4

print(f"LAG={LAG}, outcome=Financials, treatment=Technology shock")


### 5) Create PyTorch DataLoaders


In [ ]:
# PyDeepCausalML estimators build lagged sequences internally via make_lagged_sequences.
print("DataLoaders handled inside each estimator's fit() method.")


### 6) Training and Evaluation Utilities


In [ ]:
# Training utilities — each estimator records loss in history_
def final_loss(est):
    return est.history_["loss"][-1]


### 7) Define DeepSynth (Neural Synthetic Control)


In [ ]:
# DeepSynth — neural synthetic control (pydeepcausalml.timeseries.counterfactual)
print("DeepSynth imported from pydeepcausalml.")


### 8) Define CRN (Counterfactual Recurrent Network)


In [ ]:
# CRN — Counterfactual Recurrent Network with adversarial balancing
print("CRN imported from pydeepcausalml.")


### 9) Define G-Net (Deep G-Computation)


In [ ]:
# G-Net — deep G-computation
print("GNet imported from pydeepcausalml.")


### 10) Initialize and Train All Models


In [ ]:
deep_synth = counterfactual_model(
    "deepsynth", lag=LAG, hidden=32,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE, random_state=42,
)
crn = counterfactual_model(
    "crn", lag=LAG, hidden=32, lambda_adv=0.5,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE, random_state=42,
)
gnet = counterfactual_model(
    "gnet", lag=LAG, hidden=32,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE, random_state=42,
)

deep_synth.fit(data_np, outcome)
crn.fit(data_np, treatment, outcome)
gnet.fit(data_np, treatment, outcome)

print(f"DeepSynth final loss: {final_loss(deep_synth):.4f}")
print(f"CRN final loss:       {final_loss(crn):.4f}")
print(f"GNet final loss:      {final_loss(gnet):.4f}")


### 11) Evaluate Validation MSE


In [ ]:
metrics = {
    "DeepSynth": final_loss(deep_synth),
    "CRN": final_loss(crn),
    "GNet": final_loss(gnet),
}
print("\nValidation training loss (lower is better):")
for name, val in sorted(metrics.items(), key=lambda x: x[1]):
    print(f"  {name:<12} {val:.6f}")


### 12) Plot Validation Loss Curves


In [ ]:
plt.figure(figsize=(10, 4))
for name, est in [("DeepSynth", deep_synth), ("CRN", crn), ("GNet", gnet)]:
    plt.plot(est.history_["loss"], label=name, lw=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training loss curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 13) Compute Counterfactuals, ITE, and ATE


In [ ]:
ite_crn = crn.predict_ite(data_np, treatment)
ite_gnet = gnet.predict_ite(data_np)
cf_ds = deep_synth.predict_counterfactual(data_np)

ate_crn = float(np.mean(ite_crn[LAG:]))
ate_gnet = float(np.mean(ite_gnet[LAG:]))
ate_ds = float(np.mean(cf_ds) - np.mean(outcome[LAG:]))

print(f"ATE (DeepSynth proxy): {ate_ds:.4f}")
print(f"ATE (CRN):             {ate_crn:.4f}")
print(f"ATE (GNet):            {ate_gnet:.4f}")


### 14) Plot ITE Distributions


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(ite_crn[LAG:], bins=30, alpha=0.7, label="CRN")
axes[0].hist(ite_gnet[LAG:], bins=30, alpha=0.7, label="GNet")
axes[0].set_title("ITE distributions")
axes[0].set_xlabel("ITE")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(cf_ds[:200], label="DeepSynth counterfactual", lw=1.5)
axes[1].plot(outcome[LAG:LAG + 200], label="Observed outcome", lw=1.5, alpha=0.7)
axes[1].set_title("DeepSynth counterfactual vs observed")
axes[1].set_xlabel("Time index")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Interpretation Notes

- **DeepSynth** is intuitive when donor units are meaningful and stable pre-intervention comparators.
- **CRN** is useful under treatment-confounding feedback, especially with richer balancing objectives.
- **G-Net** naturally supports policy simulation by rolling trajectories under hypothetical treatment sequences.

In real applications, strengthen identification with domain assumptions (no unmeasured confounding, positivity, consistency), richer diagnostics, and robustness checks.

## Summary and Conclusion

In this notebook, we explored the estimation of individual and average treatment effects (ITE and ATE) in the context of counterfactual reasoning and causal inference, using three different neural modeling approaches: DeepSynth, CRN (Counterfactual Recurrent Network), and GNet. Each model was trained and evaluated on synthetic panel data designed to capture temporal dynamics and complex treatment-outcome relationships.

We demonstrated how to collect both factual and counterfactual predictions from each model, and compared the resulting ITE distributions and overall ATE estimates. Our visualizations showed the diversity in ITE estimates across methods, highlighting how model choice and underlying assumptions may impact causal estimations.

**Key Takeaways:**
- Counterfactual deep learning approaches can efficiently estimate individual- and population-level causal effects, even in the presence of complex time dependencies.
- Different model architectures have different strengths: DeepSynth leverages donor similarity, CRN explicitly models treatment-confounding feedback, and GNet supports flexible policy simulations.
- Proper interpretation of results requires attention to identification assumptions (such as unconfoundedness and positivity) and model diagnostics.

**Practical Recommendation:**  
When applying these models to real data, it is crucial to supplement them with rigorous domain knowledge, careful feature engineering, and robust sensitivity analyses to ensure credible causal conclusions.

## Resources

- **"Causal Inference: The Mixtape" by Scott Cunningham**  
  Great for intuitive explanations; [Online Book](https://mixtape.scunning.com/)

- **"Elements of Causal Inference" by Peters, Janzing, & Schölkopf**  
  A modern introduction to theory & algorithms; [PDF](https://mitpress.mit.edu/9780262037310/elements-of-causal-inference/)

- **"The Book of Why" by Judea Pearl**  
  A nontechnical introduction to the ladder of causation.

- **CausalML: Practical Causal Machine Learning in Python**  
  [Repo & Documentation](https://github.com/uber/causalml) – covers uplift modeling and deep ITE methods implemented in Python.

- **"DoWhy - An End-to-End Library for Causal Inference"**  
  [GitHub](https://github.com/py-why/dowhy) – library for causal inference in Python.

